In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *


items_schema = ArrayType(
    StructType(
        [
            StructField("item_id", StringType()),
            StructField("name", StringType()),
            StructField("category", StringType()),
            StructField("quantity", IntegerType()),
            StructField("unit_price", DecimalType(10, 2)),
            StructField("subtotal", DecimalType(10, 2)),
        ]
    )
)  


#Notes:
# 1. date_format(column, format)
# 2. "E" means abbreviated day of week. Ex: Mon, Tue

df_fact_orders = (
        spark.read.table("01_bronze.orders")
        .withColumn("order_timestamp", F.to_timestamp(F.col("order_timestamp")))
        .withColumn("order_date", F.to_date(F.col("order_timestamp")))
        .withColumn("order_hour", F.hour(F.col("order_timestamp")))
        .withColumn("day_of_week", F.date_format(F.col("order_timestamp"), "EEEE"))
        .withColumn(
            "is_weekend",
            F.when(
                F.date_format(F.col("order_timestamp"), "E").isin(["Sat", "Sun"]), True
            ).otherwise(False),
        )
        .withColumn("items_parsed", F.from_json(F.col("items"), items_schema))
        .withColumn("item_count", F.size(F.col("items_parsed")))
        .select(
            "order_id",
            "order_timestamp",
            "order_date",
            "order_hour",
            "day_of_week",
            "is_weekend",
            "restaurant_id",
            "customer_id",
            "order_type",
            "item_count",
            F.col("total_amount").cast("decimal(10,2)").alias("total_amount"),
            "payment_method",
            "order_status"
        )
    )

display(df_fact_orders.limit(10))